# 🔍 Qdrant Search Modes Demo

## What You'll Learn

This notebook demonstrates the **three search modes** available in Qdrant through Llama Stack, plus additional features like **ranking options**, **score thresholds**, and **vector store management**.

| Mode | How It Works | Best For |
|------|--------------|----------|
| **Vector** | Converts query to embeddings and finds semantically similar documents | Finding conceptually related content even with different words |
| **Keyword** | Splits query into words and finds documents matching any word via Qdrant's `MatchText` filter | Finding specific terms, names, or phrases |
| **Hybrid** | Performs vector similarity search filtered by keyword matches in a single Qdrant call | Semantic search narrowed by exact keyword constraints |

## Why This Matters

Different search modes excel at different tasks:
- **Vector search** understands that "automobile" and "car" mean the same thing
- **Keyword search** finds exact word matches like company names or technical terms
- **Hybrid search** combines both: vector similarity constrained by keyword filters

## Prerequisites

Before running this notebook, ensure you have:
1. ✅ Ollama running (`ollama serve`)
2. ✅ Qdrant running on port 6333 (`podman run -d -p 6333:6333 -p 6334:6334 qdrant/qdrant`)
3. ✅ Llama Stack running on port 8321 with Qdrant configured:
   ```bash
   OLLAMA_URL=http://localhost:11434/v1 QDRANT_URL=http://localhost:6333 llama stack run starter --port 8321
   ```

---
## Step 1: Connect to Llama Stack

**What we're doing:** Establishing a connection to the Llama Stack server using the OpenAI-compatible API.

**Why:** Llama Stack provides an OpenAI-compatible interface, so we can use the familiar OpenAI Python client to interact with Qdrant through Llama Stack.

**Expected output:** A success message confirming the connection.

In [ ]:
import json
import io
from pathlib import Path
from openai import OpenAI

# Connect to Llama Stack using the OpenAI client
# We point base_url to our local Llama Stack server
client = OpenAI(
    base_url="http://localhost:8321/v1/",  # Llama Stack server
    api_key="none",  # No API key needed for local server
)

print("✅ Connected to Llama Stack!")
print("   Server: http://localhost:8321")

---
## Step 2: Load the Startups Dataset

**What we're doing:** Loading a dataset of ~40,000 startup company descriptions from a JSON Lines file.

**Why:** We need real-world data to demonstrate search capabilities. This dataset contains company names, descriptions, and cities - perfect for testing different search scenarios.

**Expected output:** Dataset statistics showing how many startups were loaded.

In [ ]:
# Load the startups dataset (JSON Lines format - one JSON object per line)
data_path = Path("../data/startups_demo.json")

startups_data = []
with open(data_path, encoding="utf-8") as f:
    for line in f:
        if line.strip():  # Skip empty lines
            startups_data.append(json.loads(line))

print(f"✅ Loaded {len(startups_data):,} startup records")
print("\n📊 Sample record:")
print(json.dumps(startups_data[0], indent=2))

---
## Step 3: Create a Vector Store in Qdrant

**What we're doing:** Creating a new vector store (collection) in Qdrant through Llama Stack.

**Why:** A vector store is where we'll store our startup documents along with their vector embeddings. The embeddings allow us to perform semantic search.

**Key parameters:**
- `provider_id: "qdrant"` - Use Qdrant as the backend
- `embedding_model: "ollama/nomic-embed-text:latest"` - Use Ollama's embedding model to convert text to vectors

**Expected output:** A vector store ID that we'll use for all subsequent operations.

In [ ]:
# Create a new vector store backed by Qdrant
vector_store = client.vector_stores.create(
    name="startups_search_demo",
    extra_body={
        "provider_id": "qdrant",  # Use Qdrant as the vector database
        "embedding_model": "ollama/nomic-embed-text:latest",  # 768-dim embeddings
    },
)

print("✅ Created vector store!")
print(f"   ID: {vector_store.id}")
print(f"   Name: {vector_store.name}")

---
## Step 4: Insert Documents into the Vector Store

**What we're doing:** Taking 50 startup descriptions and inserting them into the vector store.

**Why:** Each document will be:
1. Converted to vector embeddings by the embedding model
2. Stored in Qdrant along with the original text
3. Made searchable via vector similarity, keyword matching, or both

**Process:**
1. Create a text file from each startup's data
2. Upload the file to Llama Stack
3. Attach the file to our vector store (this triggers embedding generation)

**Expected output:** Progress messages as documents are inserted (this may take 1-2 minutes).

In [ ]:
# We'll use 50 startups for a good demo (adjust for speed vs. variety)
demo_startups = startups_data[:50]
print(f"📥 Inserting {len(demo_startups)} startup documents...")
print("   (Each document is embedded and indexed in Qdrant)\n")

for i, startup in enumerate(demo_startups):
    # Create document content with company info
    content = f"""Company: {startup.get("name", "Unknown")}
City: {startup.get("city", "Unknown")}
Description: {startup.get("description", "No description available")}
"""

    # Upload as a file (Llama Stack expects files)
    pseudo_file = io.BytesIO(content.encode("utf-8"))
    uploaded_file = client.files.create(
        file=(f"startup_{i}.txt", pseudo_file, "text/plain"), purpose="assistants"
    )

    # Attach file to vector store - this triggers embedding generation
    client.vector_stores.files.create(
        vector_store_id=vector_store.id, file_id=uploaded_file.id
    )

    # Show progress every 10 documents
    if (i + 1) % 10 == 0:
        print(f"   ✓ Inserted {i + 1}/{len(demo_startups)} documents")

print(f"\n✅ All {len(demo_startups)} documents inserted and indexed!")

---
## Step 5: Define Search Helper Functions

**What we're doing:** Creating reusable functions to search with different modes and display results.

**Why:** This makes it easy to compare different search modes with the same query.

**Key parameters:**
- `search_mode`: `"vector"` (semantic), `"keyword"` (text matching), or `"hybrid"` (filtered vector)
- `score_threshold`: Minimum score to include results (0.0 to 1.0)
- `ranking_options`: Controls ranking strategy and score filtering

In [ ]:
def search(query, mode="vector", max_results=5, score_threshold=None, ranker=None):
    """
    Search the vector store with a specific mode.

    Args:
        query: The search query text
        mode: "vector", "keyword", or "hybrid"
        max_results: Maximum number of results to return
        score_threshold: Minimum score to include (0.0 to 1.0)
        ranker: Ranking strategy - "rrf", "weighted", or None
    """
    kwargs = {
        "vector_store_id": vector_store.id,
        "query": query,
        "max_num_results": max_results,
        "extra_body": {"search_mode": mode},
    }

    # Build ranking_options if score_threshold or ranker is specified
    if score_threshold is not None or ranker is not None:
        ranking_options = {}
        if score_threshold is not None:
            ranking_options["score_threshold"] = score_threshold
        if ranker is not None:
            ranking_options["ranker"] = ranker
        kwargs["ranking_options"] = ranking_options

    results = client.vector_stores.search(**kwargs)
    return results


def display_results(results, title):
    """
    Display search results in a readable format.
    """
    print(f"\n{'=' * 70}")
    print(f"🔍 {title}")
    print(f"{'=' * 70}")

    if not results.data:
        print("   ❌ No results found")
        return

    for i, result in enumerate(results.data, 1):
        # Extract company name from content
        content = result.content[0].text if result.content else "N/A"
        lines = content.split("\n")
        company = lines[0].replace("Company: ", "") if lines else "Unknown"

        print(f"\n   {i}. {company}")
        print(f"      Score: {result.score:.4f}")
        # Show first 150 chars of description
        desc_start = content.find("Description:")
        if desc_start > 0:
            desc = content[desc_start + 12 : desc_start + 162].strip()
            print(f"      {desc}...")


print("✅ Helper functions defined!")

---
## Step 6: Vector Search (Semantic Similarity)

**What we're doing:** Searching using vector embeddings to find semantically similar content.

**How it works:**
1. Your query is converted to a vector embedding
2. Qdrant finds documents whose vectors are most similar (cosine similarity)
3. Results are ranked by similarity score

**Why it's powerful:** Vector search understands meaning, not just words. It can find:
- "automobile" when you search for "car"
- "AI healthcare" when you search for "medical machine learning"

**Expected output:** Companies semantically related to the query, even if they don't contain the exact words.

In [ ]:
# Let's try a semantic query
query = "artificial intelligence for healthcare"
print(f"📝 Query: '{query}'")
print("\n💡 Vector search will find companies related to AI + healthcare,")
print("   even if they use different terminology.")

results = search(query, mode="vector")
display_results(results, "VECTOR SEARCH - Semantic Similarity")

In [ ]:
# Try another semantic query with different phrasing
query = "mobile apps for consumers"
print(f"📝 Query: '{query}'")
print("\n💡 This should find mobile/consumer tech companies,")
print("   including those that say 'smartphone' or 'users' instead.")

results = search(query, mode="vector")
display_results(results, "VECTOR SEARCH - Mobile/Consumer")

---
## Step 7: Keyword Search (Text Matching)

**What we're doing:** Searching for text matches in document content using Qdrant's `MatchText` filter.

**How it works:**
1. The query is split into individual words (lowercased)
2. Qdrant's `scroll` method searches for documents matching **any** of those words (`should` / OR logic)
3. Matching documents receive a fixed score of 1.0 (no similarity ranking)
4. Results are filtered by `score_threshold`

**When to use keyword search:**
- Finding specific company names or technical terms
- When you need literal word matching
- When semantic similarity is not important

**Key differences from vector search:**
- No embedding generation needed (faster for simple lookups)
- Matches are on exact words, not meaning
- All matching results get the same score (1.0)

In [ ]:
# Keyword search for a specific term
query = "software"
print(f"📝 Query: '{query}'")
print("\n💡 Keyword search will ONLY find documents that")
print("   contain the exact word 'software'.")

results = search(query, mode="keyword")
display_results(results, "KEYWORD SEARCH - Exact Match")

In [ ]:
# Compare: Vector search for the same term
print("📝 Same query with VECTOR search for comparison:")

results = search("software", mode="vector")
display_results(results, "VECTOR SEARCH - Same Query 'software'")

print("\n💡 Notice: Vector search may find related companies even")
print("   without the exact word 'software' in their description.")

---
## Step 8: Hybrid Search (Filtered Vector Similarity)

**What we're doing:** Performing vector similarity search filtered by keyword matches.

**How it works:**
1. The query words are extracted and used to build a `MatchText` filter (OR / `should` logic)
2. A single Qdrant `query_points` call combines the vector query with this keyword filter
3. Results must match **at least one keyword** AND are ranked by **vector similarity**
4. Scores come from cosine similarity (same as vector search), but the candidate set is narrowed

**Why this is useful:**
- **More precise** than pure vector search -- results must contain relevant words
- **Ranked by meaning** unlike pure keyword search -- most semantically relevant first
- **Efficient** -- single Qdrant call, not two separate searches

**Important:** Unlike some hybrid implementations that fuse separate vector + keyword results (e.g., RRF), Qdrant's approach applies keyword conditions as a **pre-filter** on vector search. This means results will always contain at least one query word.

**Expected output:** Semantically ranked results that also contain keyword matches.

In [ ]:
query = "fintech payment processing"
print(f"📝 Query: '{query}'")
print("\n💡 Hybrid search performs vector similarity FILTERED by keywords:")
print(
    "   • Only documents containing 'fintech', 'payment', or 'processing' are candidates"
)
print("   • Those candidates are ranked by cosine similarity to the query embedding")
print("   • Scores are vector similarity scores (like vector search)")

results = search(query, mode="hybrid")
display_results(results, "HYBRID SEARCH - Filtered Vector Similarity")

---
## Step 9: Side-by-Side Comparison

**What we're doing:** Comparing all three search modes with the same query.

**Why:** This clearly shows how each mode behaves differently and helps you understand which to use for your use case.

**What to look for:**
- **Vector**: Broad semantic matches ranked by similarity, may not contain exact query words
- **Keyword**: Only word-level matches (OR logic), all scored equally at 1.0
- **Hybrid**: Vector similarity results filtered to those containing at least one query word

In [ ]:
# Compare all three modes with the same query
comparison_query = "data analytics platform"

print(f"\n{'#' * 70}")
print("# COMPARISON: All Search Modes")
print(f"# Query: '{comparison_query}'")
print(f"{'#' * 70}")

for mode in ["vector", "keyword", "hybrid"]:
    results = search(comparison_query, mode=mode, max_results=3)
    display_results(results, f"{mode.upper()} SEARCH")

---
## Step 10: Ranking Options & Score Thresholds

**What we're doing:** Using `ranking_options` to control result quality via score thresholds.

**How it works:**
- `score_threshold` filters out results below a minimum similarity score
- For **vector** and **hybrid** search, scores are cosine similarity (0.0 to 1.0)
- For **keyword** search, all matches get a fixed score of 1.0

**Why this matters:** In production, you'll want to filter out low-quality matches to avoid returning irrelevant results.

In [ ]:
query = "machine learning analytics"

# Search WITHOUT score threshold (returns all matches)
results_all = search(query, mode="vector", max_results=5)
print(f"📝 Query: '{query}'\n")
print("WITHOUT score_threshold (default 0.0):")
display_results(results_all, "VECTOR SEARCH - No threshold")

# Search WITH a high score threshold
results_filtered = search(query, mode="vector", max_results=5, score_threshold=0.55)
print("\nWITH score_threshold=0.55 (only high-confidence matches):")
display_results(results_filtered, "VECTOR SEARCH - threshold=0.55")

print("\n💡 Notice: Setting a higher score_threshold filters out weaker matches,")
print("   giving you only the most relevant results.")

---
## Step 11: Vector Store Management

**What we're doing:** Demonstrating the OpenAI-compatible CRUD operations for vector stores and files.

**Available operations:**
- **List** vector stores
- **Retrieve** a specific vector store
- **List files** in a vector store
- **Update** vector store metadata

In [ ]:
# List all vector stores
stores = client.vector_stores.list()
print(f"📋 Vector Stores ({len(stores.data)} total):")
for vs in stores.data:
    print(f"   • {vs.id}  name={vs.name}  files={vs.file_counts}")

# Retrieve our specific vector store
retrieved = client.vector_stores.retrieve(vector_store.id)
print("\n🔎 Retrieved vector store details:")
print(f"   ID:     {retrieved.id}")
print(f"   Name:   {retrieved.name}")
print(f"   Status: {retrieved.status}")
print(f"   Files:  {retrieved.file_counts}")

# List files in the vector store
files = client.vector_stores.files.list(vector_store.id)
print(f"\n📁 Files in vector store ({len(files.data)} total):")
for f in files.data[:5]:
    print(f"   • {f.id}  status={f.status}")
if len(files.data) > 5:
    print(f"   ... and {len(files.data) - 5} more")

In [ ]:
# Update vector store metadata
updated = client.vector_stores.update(
    vector_store.id,
    name="startups_search_demo_updated",
    metadata={"demo": "search_modes", "version": "2"},
)
print("✏️  Updated vector store:")
print(f"   New name: {updated.name}")
print(f"   Metadata: {updated.metadata}")

---
## Step 12: Cleanup

**What we're doing:** Deleting the vector store to free up resources.

**Why:** Good practice to clean up test data. In production, you'd keep your vector stores.

In [ ]:
# Clean up - delete the vector store
client.vector_stores.delete(vector_store.id)
print(f"🗑️  Deleted vector store: {vector_store.id}")
print("\n✅ Demo complete!")

---
## 📚 Summary

### Search Modes

| Search Mode | How It Works | Scores | Best For |
|-------------|--------------|--------|----------|
| **Vector** | Embeds query, finds most similar vectors via cosine similarity | 0.0–1.0 (cosine) | Semantic understanding, finding related content |
| **Keyword** | Splits query into words, matches any word via `MatchText` filter | Fixed 1.0 | Exact word lookup, specific terms/names |
| **Hybrid** | Vector similarity search filtered by keyword `MatchText` conditions | 0.0–1.0 (cosine) | Semantic search narrowed by keyword constraints |

### Additional Features Demonstrated

| Feature | Description |
|---------|-------------|
| **Score threshold** | Filter out low-confidence results via `ranking_options` |
| **Vector store CRUD** | Create, list, retrieve, update, and delete vector stores |
| **File management** | Upload files, attach to stores, list files in a store |

### Key Takeaways

1. **Vector search** is powerful for understanding intent and finding related content
2. **Keyword search** is useful when you need literal word matching without embeddings
3. **Hybrid search** combines both: vector ranking with keyword pre-filtering in a single Qdrant call
4. **Score thresholds** help you control result quality in production
5. **OpenAI-compatible API** makes it easy to manage vector stores and files

### Next Steps

- Try `02_multilingual_demo.ipynb` for cross-language search
- Try `03_multimodal_demo.ipynb` for image + text search
- Try `04_advanced_features_demo.ipynb` for filtering and advanced features